# 3. Merging

Merging joins two branches back together. There are three cases:

1. **Fast-forward** — nothing to join, git just moves the branch forward.
2. **Merge commit** — both branches changed, git combines them.
3. **Conflict** — both branches changed the same line, and you decide.

We start from three branches, each changing a different part of the file:

```mermaid
gitGraph
    commit id: "volume"
    branch froude
    commit id: "froude number"
    checkout main
    branch report
    commit id: "units"
    checkout main
    branch fresh-water
    commit id: "fresh water"
```

In [1]:
import os, shutil, stat, sys
from pathlib import Path


def remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)  # git makes its object files read-only
    func(path)


work = Path.home() / "git-course" / "03-merging"
if work.exists():
    shutil.rmtree(work, onexc=remove_readonly)
work.mkdir(parents=True)
%cd {work}

/home/emircem/git-course/03-merging


## Starting point

In [2]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

Writing hydrostatics.py


In [3]:
!git init
!git add hydrostatics.py
!git commit -m "Add volume and displacement for a box barge"

Initialized empty Git repository in /home/emircem/git-course/03-merging/.git/
[main (root-commit) 711ecea] Add volume and displacement for a box barge
 1 file changed, 13 insertions(+)
 create mode 100644 hydrostatics.py


Branch `froude` adds a function at the end of the file:

In [4]:
!git switch -c froude

Switched to a new branch 'froude'


In [5]:
%%writefile -a hydrostatics.py


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5

Appending to hydrostatics.py


In [6]:
!git add hydrostatics.py
!git commit -m "Add the Froude number"

[froude 328671d] Add the Froude number
 1 file changed, 5 insertions(+)


Branch `report` changes the first line. `git switch -c report main` means
"create `report`, starting from `main`".

In [7]:
!git switch -c report main

Switched to a new branch 'report'


In [8]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge, in SI units (m, kg, N)."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

Overwriting hydrostatics.py


In [9]:
!git add hydrostatics.py
!git commit -m "Say which units the module uses"

[report 4a9c92c] Say which units the module uses
 1 file changed, 1 insertion(+), 1 deletion(-)


Branch `fresh-water` changes the density:

In [10]:
!git switch -c fresh-water main

Switched to a new branch 'fresh-water'


In [11]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge."""

RHO = 1000.0  # kg/m^3, fresh water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)

Overwriting hydrostatics.py


In [12]:
!git add hydrostatics.py
!git commit -m "Use fresh water density for river barges"

[fresh-water e94bf77] Use fresh water density for river barges
 1 file changed, 1 insertion(+), 1 deletion(-)


In [13]:
!git switch main
!git log --oneline --graph --all

Switched to branch 'main'
* e94bf77 (fresh-water) Use fresh water density for river barges
| * 4a9c92c (report) Say which units the module uses
|/  
| * 328671d (froude) Add the Froude number
|/  
* 711ecea (HEAD -> main) Add volume and displacement for a box barge


## Case 1: fast-forward

Nothing has happened on `main` since `froude` started. So git does not
need to combine anything — it simply moves `main` forward.

In [14]:
!git merge froude

Updating 711ecea..328671d
Fast-forward
 hydrostatics.py | 5 +++++
 1 file changed, 5 insertions(+)


In [15]:
!git log --oneline --graph --all

* e94bf77 (fresh-water) Use fresh water density for river barges
| * 4a9c92c (report) Say which units the module uses
|/  
| * 328671d (HEAD -> main, froude) Add the Froude number
|/  
* 711ecea Add volume and displacement for a box barge


`main` and `froude` now point at the same commit. The branch has done its
job:

In [16]:
!git branch -d froude

Deleted branch froude (was 328671d).


## Case 2: a merge commit

First, `main` gets a new commit of its own:

In [17]:
%%writefile -a hydrostatics.py


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0

Appending to hydrostatics.py


In [18]:
!git add hydrostatics.py
!git commit -m "Add tonnes per centimetre immersion"

[main b91cbf5] Add tonnes per centimetre immersion
 1 file changed, 5 insertions(+)


Now `main` and `report` have both moved. Git combines them in a new commit
with **two parents**:

```mermaid
gitGraph
    commit id: "volume"
    commit id: "froude number"
    branch report
    commit id: "units"
    checkout main
    commit id: "tonnes per cm"
    merge report
```

In [19]:
!git merge --no-edit report

Auto-merging hydrostatics.py
Merge made by the 'ort' strategy.
 hydrostatics.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [20]:
!git log --oneline --graph --all

*   84cc5b8 (HEAD -> main) Merge branch 'report'
|\  
| * 4a9c92c (report) Say which units the module uses
* | b91cbf5 Add tonnes per centimetre immersion
* | 328671d Add the Froude number
|/  
| * e94bf77 (fresh-water) Use fresh water density for river barges
|/  
* 711ecea Add volume and displacement for a box barge


The two branches changed different lines, so git merged them by itself.
Most merges are like this.

In [21]:
print(Path("hydrostatics.py").read_text())

"""Hydrostatics for a box-shaped barge, in SI units (m, kg, N)."""

RHO = 1025.0  # kg/m^3, sea water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0



In [22]:
!git branch -d report

Deleted branch report (was 4a9c92c).


## Case 3: a conflict

On `main`, someone sets the density to the ITTC standard value, 1026:

In [23]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge, in SI units (m, kg, N)."""

RHO = 1026.0  # kg/m^3, sea water (ITTC standard)


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0

Overwriting hydrostatics.py


In [24]:
!git add hydrostatics.py
!git commit -m "Use the ITTC standard sea water density"

[main d01b27b] Use the ITTC standard sea water density
 1 file changed, 1 insertion(+), 1 deletion(-)


But `fresh-water` changed the same line to 1000. Git cannot choose for you:

In [25]:
# This one fails on purpose: a conflict.
!git merge fresh-water

Auto-merging hydrostatics.py
CONFLICT (content): Merge conflict in hydrostatics.py
Automatic merge failed; fix conflicts and then commit the result.


Nothing is broken. The merge is paused, and `git status` says so:

In [26]:
!git status

On branch main
You have unmerged paths.
  (fix conflicts and run "git commit")
  (use "git merge --abort" to abort the merge)

Unmerged paths:
  (use "git add <file>..." to mark resolution)
	both modified:   hydrostatics.py

no changes added to commit (use "git add" and/or "git commit -a")


Git wrote both versions into the file:

In [27]:
print(Path("hydrostatics.py").read_text())

"""Hydrostatics for a box-shaped barge, in SI units (m, kg, N)."""

<<<<<<< HEAD
RHO = 1026.0  # kg/m^3, sea water (ITTC standard)
RHO = 1000.0  # kg/m^3, fresh water
>>>>>>> fresh-water


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0



- Between `<<<<<<< HEAD` and `=======` is **our** version (`main`).
- Between `=======` and `>>>>>>> fresh-water` is **their** version.

### The escape hatch

Not ready to decide? `git merge --abort` undoes the whole merge:

In [28]:
!git merge --abort
!git status

On branch main
nothing to commit, working tree clean


Back to normal. Now let's do it for real:

In [29]:
# This one fails on purpose: the same conflict again.
!git merge fresh-water

Auto-merging hydrostatics.py
CONFLICT (content): Merge conflict in hydrostatics.py
Automatic merge failed; fix conflicts and then commit the result.


### Resolving

Edit the file so it is right, and remove the marker lines. You don't have
to pick a side — here we keep both densities:

In [30]:
%%writefile hydrostatics.py
"""Hydrostatics for a box-shaped barge, in SI units (m, kg, N)."""

RHO_SEA = 1026.0  # kg/m^3, sea water (ITTC standard)
RHO_FRESH = 1000.0  # kg/m^3, fresh water
RHO = RHO_SEA


def volume(length, beam, draft):
    """Immersed volume of the hull, in m^3."""
    return length * beam * draft


def displacement(length, beam, draft):
    """Mass of the water the hull pushes aside, in kg."""
    return RHO * volume(length, beam, draft)


def froude_number(speed, length):
    """Speed-length ratio that scales wave resistance."""
    return speed / (9.81 * length) ** 0.5


def tonnes_per_cm(length, beam):
    """Tonnes of cargo that sink the barge by one centimetre."""
    return length * beam * RHO / 100000.0

Overwriting hydrostatics.py


`git add` marks the file as resolved, and `git commit` finishes the merge.
`--no-edit` keeps the message git already prepared.

In [31]:
!git add hydrostatics.py
!git commit --no-edit

[main e45956b] Merge branch 'fresh-water'


In [32]:
!git log --oneline --graph --all

*   e45956b (HEAD -> main) Merge branch 'fresh-water'
|\  
| * e94bf77 (fresh-water) Use fresh water density for river barges
* | d01b27b Use the ITTC standard sea water density
* |   84cc5b8 Merge branch 'report'
|\ \  
| * | 4a9c92c Say which units the module uses
| |/  
* | b91cbf5 Add tonnes per centimetre immersion
* | 328671d Add the Froude number
|/  
* 711ecea Add volume and displacement for a box barge


In [33]:
!git branch -d fresh-water

Deleted branch fresh-water (was e94bf77).


## Summary

| I want to… | Command |
| - | - |
| merge a branch into the current one | `git merge <branch>` |
| give up on a merge | `git merge --abort` |
| mark a conflict as resolved | `git add <file>` |
| finish the merge | `git commit --no-edit` |

Merge often: small merges rarely conflict.

## Exercises

1. Make two branches that change the same line, merge them, and resolve
   the conflict.
2. Start a merge with a conflict and get out of it with `git merge --abort`.

Next: [4. Rebase](04-rebase.ipynb)